# 3.1.2 斜率的假设检验
作者：林泓浩

使用虚构拟合汇总量：n=10、斜率估计1.2、标准误0.4。这里没有原始数据，也没有训练步骤。与手稿斜率为1的算例不同。本 Notebook 可独立从头运行，依次执行四个代码单元即可。
在 VS Code 中选择安装了 SciPy 的 Python 内核。图示未在本实验新增。


## 代码 3.1.2-07 · 导入 t 分布并明确检验输入


In [1]:
from scipy.stats import t

n = 10
beta1_hat = 1.2
slope_se = 0.4
beta1_null = 0.0
alpha = 0.05
df = n - 2


### 逐步读代码
- `from scipy.stats import t` 从 SciPy 的统计模块导入 t 分布对象。`t` 用来调用分布函数，不是本次统计量的数值；后者将存入 `t_obs`。
- `n` 是样本量；`beta1_hat` 是拟合得到的斜率估计；`slope_se` 是斜率的估计标准误。本例直接给定这些汇总量，所以没有读取数据或训练模型。
- `beta1_null = 0.0` 保存原假设规定的斜率值；`alpha = 0.05` 保存事先选定的显著性水平。赋值符号 `=` 把右边的值交给左边的变量。
- `df = n - 2` 计算简单线性回归的残差自由度，结果为整数 8；模型估计了截距和斜率两个参数。其余带小数点的输入是浮点标量，不是数组。


## 代码 3.1.2-08 · 把估计值到假设值的距离标准化


In [2]:
t_obs = (beta1_hat - beta1_null) / slope_se
print(f"t = {t_obs:.6f}")


t = 3.000000


### 逐步读代码
- 括号中的 `beta1_hat - beta1_null` 先计算估计值到原假设值的带符号距离，再用 `/ slope_se` 除以估计标准误。本例是 (1.2 - 0) / 0.4，得到浮点标量 3。
- `t_obs` 保存这份数据对应的统计量观测值。若斜率估计小于原假设值，结果可以为负；双侧检验下一步会取绝对值。
- `print` 把结果显示出来。`f"..."` 是格式化字符串，花括号插入变量值；`:.6f` 表示以小数形式显示六位小数。它只改变显示方式，不会把变量本身提前截断。


## 代码 3.1.2-09 · 计算双侧 p 值


In [3]:
right_tail = t.sf(abs(t_obs), df=df)
p_value = 2 * right_tail
print(f"right tail = {right_tail:.6f}")
print(f"p = {p_value:.6f}")


right tail = 0.008536
p = 0.017072


### 逐步读代码
- `abs(t_obs)` 取统计量的绝对值，使正向与负向同样大的偏离得到相同处理。本例的 3 和 -3 都会变成 3。
- `t.sf(x, df=8)` 是 t 分布的生存函数（survival function），给出 P(T > x) 的右尾概率；连续分布下它也等于 P(T ≥ x)。已知一个坐标，`sf` 返回该坐标右侧的面积。
- `df=df` 左边是函数的参数名，右边是上一块定义的变量；这里相当于把自由度 8 传入。`right_tail` 保存右尾概率，是一个数值标量。
- `p_value = 2 * right_tail` 用对称性把右尾面积乘二，得到双侧 p 值。必须先取绝对值；对负的统计量直接求右尾再乘二可能得到超过 1 的错误结果。
- 两次 `print` 分别显示单侧右尾面积与双侧 p 值，仍使用六位小数；最终比较应使用未格式化的 `p_value`。


## 代码 3.1.2-10 · 作出判断并用置信区间核对


In [4]:
reject_null = p_value < alpha
critical_value = t.ppf(1 - alpha / 2, df=df)
margin = critical_value * slope_se
ci_lower = beta1_hat - margin
ci_upper = beta1_hat + margin
print("Reject H0:", reject_null)
print(f"CI = [{ci_lower:.6f}, {ci_upper:.6f}]")


Reject H0: True
CI = [0.277598, 2.122402]


### 逐步读代码
- `p_value < alpha` 是大小比较，结果为布尔值 `True` 或 `False`，保存为 `reject_null`。`True` 表示按当前规则拒绝原假设，`False` 表示不拒绝；后者并不证明原假设正确。
- `t.ppf(1 - alpha / 2, df=df)` 输入左侧累计概率并返回对应分位数。当前输入 0.975 和自由度 8，得到右端临界值约 2.306004。`ppf` 是“概率找坐标”，上一块的 `sf` 是“坐标找右尾概率”。
- `margin = critical_value * slope_se` 用乘法 `*` 求区间半宽。`ci_lower` 从斜率估计中减去半宽，`ci_upper` 加上半宽；得到的是与当前显著性水平对应的双侧置信区间。
- 第一个 `print` 依次显示提示文字和布尔结果；第二个把两个端点插入方括号中。这只是显示区间的字符串，不是在这里创建 Python 列表。
- 如果事先把显著性水平定为 0.01，这份数据的 p 值仍约为 0.017072，判断会变为不拒绝；对应的区间改成 99% 区间。alpha 决定判断门槛及对应区间，不参与上述 p 值公式。
